# 🧪 Notebook 3: Full Knowledge Distillation on DeepCrack Dataset
This notebook evaluates the SAM 2 to YOLO Knowledge Distillation framework trained **exclusively on DeepCrack**, featuring the **Segmentation Head Freeze Ablation**.

### Experiments in this Notebook:
1. `v8_dc_baseline`: YOLOv8n-seg baseline (no KD) on DeepCrack
2. `v8_dc_full_kd_box`: YOLOv8n-seg with Full KD (Box prompts) on DeepCrack
3. `v11_dc_baseline`: YOLOv11n-seg baseline (no KD) on DeepCrack
4. `v11_dc_full_kd_box`: YOLOv11n-seg with Full KD (Box prompts) on DeepCrack
5. `v11_dc_seghead_frozen_kd`: **NEW**: YOLOv11n-seg with Segment Head Frozen throughout full KD run


In [ ]:
!mkdir -p configs utils distillation scripts checkpoints data/datasets data/teacher_logits_box data/teacher_logits_centroid


In [ ]:
# Install dependencies and SAM 2
!pip install -q ultralytics albumentations pycocotools thop pyyaml
!git clone https://github.com/facebookresearch/sam2.git sam2_repo || true
%cd sam2_repo
!pip install -e .
%cd ..
!rm -rf sam2


In [ ]:
import os
import shutil
from pathlib import Path

input_dir = Path("/kaggle/input/distill_datasetforme")
if not input_dir.exists():
    input_dir = Path("/kaggle/input")

datasets_dir = Path("data/datasets")
datasets_dir.mkdir(parents=True, exist_ok=True)
checkpoints_dir = Path("checkpoints")
checkpoints_dir.mkdir(parents=True, exist_ok=True)

# Link teacher logits to /tmp
for folder in ["teacher_logits_box", "teacher_logits_centroid", "teacher_features"]:
    p_local = Path("data") / folder
    p_tmp = Path("/tmp") / folder
    if os.path.lexists(p_local):
        os.unlink(p_local) if os.path.islink(p_local) else shutil.rmtree(p_local)
    p_tmp.mkdir(parents=True, exist_ok=True)
    os.symlink(p_tmp, p_local)

# Link SAM 2 checkpoint
for root, dirs, files in os.walk(str(input_dir)):
    if "sam2_hiera_large.pt" in files:
        dest = checkpoints_dir / "sam2_hiera_large.pt"
        if os.path.lexists(dest): os.remove(dest)
        os.symlink(Path(root) / "sam2_hiera_large.pt", dest)
        break
if not (checkpoints_dir / "sam2_hiera_large.pt").exists():
    !wget -q https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt -O checkpoints/sam2_hiera_large.pt

# Link DeepCrack
for root, dirs, files in os.walk(str(input_dir)):
    if "train_img" in dirs:
        dest = datasets_dir / "deepcrack"
        if os.path.lexists(dest): os.unlink(dest) if os.path.islink(dest) else shutil.rmtree(dest)
        os.symlink(Path(root), dest)
        break


In [ ]:
# Convert DeepCrack to YOLO format & generate teacher logits
!python scripts/convert_deepcrack.py --src data/datasets/deepcrack --dst data/datasets/deepcrack_yolo

print("=== Generating SAM 2 Box-Only Logits for DeepCrack ===")
!python scripts/generate_teacher_logits.py --prompt-type box --logits-dir data/teacher_logits_box --dataset data/datasets/deepcrack_yolo


## 🏋️ Train DeepCrack KD Models (Including SegHead Freeze Ablation)


In [ ]:
# Train comparative models on DeepCrack
!python scripts/run_experiments.py --exp baseline_finetune --cfg configs/config.yaml
!python scripts/run_experiments.py --exp full_kd_box --cfg configs/config.yaml
!python scripts/run_experiments.py --exp full_kd_seghead_frozen --cfg configs/config.yaml


## 📊 Evaluation & Head Freeze Analysis


In [ ]:
print("=== In-Domain DeepCrack Comparative Evaluation ===")
models = {
    "Baseline (No KD)": "runs/crack_distill_baseline_finetune_instance_seg_yolo11n-seg/weights/best.pt",
    "Full KD (Box Prompts)": "runs/crack_distill_full_kd_box_instance_seg_yolo11n-seg/weights/best.pt",
    "Full KD (SegHead Frozen)": "runs/crack_distill_full_kd_seghead_frozen_instance_seg_yolo11n-seg/weights/best.pt",
}

for name, path in models.items():
    print(f"\n--- Evaluating {name} ---")
    !python scripts/test_model.py --val --model {path} --data data/datasets/deepcrack_yolo/dataset.yaml
